In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

### CONFIG

In [13]:
DL_FEATURES_FILE = "../image_features_deeplearning.csv"
EPIGENETIC_FILE = "../combined_epigenetic_score_matrix.csv"
MUTATION_INTERACTION_FILE = "../mutation_epigenetic_interaction_matrix.csv"
MUTATION_PLUS_EPI_FILE = "../mutations_plus_epigenetics.csv"
CLINICAL_FILE = "../csv/Pathology_Cohorts_v7.csv"
COMMON_SAMPLES_FILE = "../csv/common_82_samples.txt"

OUTPUT_PATCH_FEATURES = "../output_features/patient_image_features_pca.csv"
OUTPUT_INTEGRATED = "../output_features/integrated_multimodal_features.csv"

N_PCA_COMPONENTS = 82  # Reduced from 2048 to 82

## IMAGE AGGREGATION

### PATCH AGGREGATION

In [14]:
print("\n" + "="*60)
print("PATCH AGGREGATION (Per Patient)")
print("="*60)

# Load image features
print(f"\nLoading image features from {DL_FEATURES_FILE}...")
img_df = pd.read_csv(DL_FEATURES_FILE)
print(f"Shape: {img_df.shape}")
print(f"Columns: {img_df.columns[:10].tolist()}...")
print(f"Unique patients: {img_df['patient_id'].nunique()}")
print(f"Unique slides: {img_df['slide_id'].nunique()}")

# Get feature columns
feature_cols = [c for c in img_df.columns if c.startswith('feat_')]
print(f"Feature columns: {len(feature_cols)}")

# Extract features and patient IDs
X_patches = img_df[feature_cols].values
patient_ids = img_df['patient_id'].values

print(f"\nPatch features matrix: {X_patches.shape}")
print(f"Patient IDs: {len(patient_ids)}")

# Aggregate per patient using MULTIPLE pooling strategies
print("\nAggregating patches per patient...")
print("Using: Mean, Max, and Median pooling")

unique_patients = sorted(img_df['patient_id'].unique())
print(f"Number of unique patients: {len(unique_patients)}")

patient_features = {}

for patient in unique_patients:
    mask = patient_ids == patient
    patient_patches = X_patches[mask]
    
    # Multiple aggregation strategies
    mean_feat = np.mean(patient_patches, axis=0)
    max_feat = np.max(patient_patches, axis=0)
    median_feat = np.median(patient_patches, axis=0)
    std_feat = np.std(patient_patches, axis=0)  # Heterogeneity measure
    
    # Concatenate all strategies
    combined = np.concatenate([mean_feat, max_feat, median_feat, std_feat])
    patient_features[patient] = combined

# Create patient-level DataFrame
patient_img_df = pd.DataFrame(patient_features).T
patient_img_df.index.name = 'patient_id'

# Name columns
n_orig = len(feature_cols)
columns = []
for agg in ['mean', 'max', 'median', 'std']:
    for i in range(n_orig):
        columns.append(f'img_{agg}_{i:04d}')

patient_img_df.columns = columns
patient_img_df = patient_img_df.reset_index()

print(f"\nPatient-level image features shape: {patient_img_df.shape}")
print(f"Total features per patient: {patient_img_df.shape[1] - 1}")  # -1 for patient_id
print(f"(= {n_orig} original features × 4 aggregation methods)")


PATCH AGGREGATION (Per Patient)

Loading image features from ../image_features_deeplearning.csv...
Shape: (77672, 2053)
Columns: ['patient_id', 'slide_id', 'tile_x', 'tile_y', 'tile_idx', 'feat_0000', 'feat_0001', 'feat_0002', 'feat_0003', 'feat_0004']...
Unique patients: 82
Unique slides: 82
Feature columns: 2048

Patch features matrix: (77672, 2048)
Patient IDs: 77672

Aggregating patches per patient...
Using: Mean, Max, and Median pooling
Number of unique patients: 82

Patient-level image features shape: (82, 8193)
Total features per patient: 8192
(= 2048 original features × 4 aggregation methods)


### PCA DIMENSIONALITY REDUCTION

In [15]:
print("\n" + "="*60)
print("PCA DIMENSIONALITY REDUCTION")
print("="*60)
# Prepare data for PCA
X_patient = patient_img_df.drop('patient_id', axis=1).values
print(f"\nInput shape: {X_patient.shape}")
print(f"  Patients: {X_patient.shape[0]}")
print(f"  Features: {X_patient.shape[1]}")

# Impute any missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_patient)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# CRITICAL FIX: PCA components cannot exceed min(n_samples, n_features)
max_components = min(X_scaled.shape[0], X_scaled.shape[1])
print(f"\nMaximum possible PCA components: {max_components}")

# Use 50 components (good balance - captures most variance, avoids overfitting)
print(f"Using: {N_PCA_COMPONENTS} components")

# Apply PCA
pca = PCA(n_components=N_PCA_COMPONENTS)
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print(f"\n{'='*50}")
print(f"PCA RESULTS")
print(f"{'='*50}")
print(f"  Original dimensions: {X_patient.shape[1]}")
print(f"  Reduced dimensions: {N_PCA_COMPONENTS}")
print(f"  Total variance explained: {cumulative_var[-1]:.2%}")
print(f"\n  Cumulative variance by components:")
for threshold in [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]:
    n_needed = np.argmax(cumulative_var >= threshold) + 1
    if n_needed > 0:
        print(f"    {threshold:.0%} variance: {n_needed} components")
    else:
        print(f"    {threshold:.0%} variance: >{max_components} components")

print(f"\n  Top 15 components individual variance:")
for i in range(min(15, N_PCA_COMPONENTS)):
    print(f"    PC{i+1:2d}: {explained_var[i]:.4f} ({explained_var[i]:.2%})")

# Create PCA DataFrame
pca_columns = [f'img_pca_{i:03d}' for i in range(N_PCA_COMPONENTS)]
pca_df = pd.DataFrame(X_pca, columns=pca_columns)
pca_df.insert(0, 'patient_id', patient_img_df['patient_id'].values)

print(f"\nFinal PCA features shape: {pca_df.shape}")

# Save
pca_df.to_csv(OUTPUT_PATCH_FEATURES, index=False)
print(f"✓ Saved to: {OUTPUT_PATCH_FEATURES}")



PCA DIMENSIONALITY REDUCTION

Input shape: (82, 8192)
  Patients: 82
  Features: 8192

Maximum possible PCA components: 82
Using: 82 components

PCA RESULTS
  Original dimensions: 8192
  Reduced dimensions: 82
  Total variance explained: 100.00%

  Cumulative variance by components:
    50% variance: 6 components
    70% variance: 16 components
    80% variance: 27 components
    90% variance: 46 components
    95% variance: 61 components
    99% variance: 76 components

  Top 15 components individual variance:
    PC 1: 0.2171 (21.71%)
    PC 2: 0.1084 (10.84%)
    PC 3: 0.0615 (6.15%)
    PC 4: 0.0514 (5.14%)
    PC 5: 0.0428 (4.28%)
    PC 6: 0.0379 (3.79%)
    PC 7: 0.0323 (3.23%)
    PC 8: 0.0242 (2.42%)
    PC 9: 0.0213 (2.13%)
    PC10: 0.0201 (2.01%)
    PC11: 0.0185 (1.85%)
    PC12: 0.0161 (1.61%)
    PC13: 0.0151 (1.51%)
    PC14: 0.0144 (1.44%)
    PC15: 0.0135 (1.35%)

Final PCA features shape: (82, 83)
✓ Saved to: ../output_features/patient_image_features_pca.csv


In [16]:
print("\n" + "="*60)
print("BONUS: ALTERNATIVE METHODS COMPARISON")
print("="*60)

# Method 1: Truncated SVD (faster for sparse/high-dim data)
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=N_PCA_COMPONENTS, random_state=42)
X_svd = svd.fit_transform(X_scaled)
print(f"\nTruncatedSVD - Variance explained: {svd.explained_variance_ratio_.sum():.2%}")

# Method 2: Feature Agglomeration (cluster similar features)
from sklearn.cluster import FeatureAgglomeration
try:
    agglo = FeatureAgglomeration(n_clusters=N_PCA_COMPONENTS)
    X_agglo = agglo.fit_transform(X_scaled.T).T  # Transpose because it clusters features
    print(f"Feature Agglomeration - Clusters created: {N_PCA_COMPONENTS}")
except:
    print("Feature Agglomeration skipped (too large)")

# Method 3: Variance Threshold (keep top features by variance)
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold()
X_var = selector.fit_transform(X_scaled)
n_selected = X_var.shape[1]
print(f"Variance Threshold - Features kept: {n_selected} (removed {X_scaled.shape[1] - n_selected} zero-variance)")


BONUS: ALTERNATIVE METHODS COMPARISON

TruncatedSVD - Variance explained: 100.00%
Feature Agglomeration - Clusters created: 82
Variance Threshold - Features kept: 7796 (removed 396 zero-variance)


In [17]:
print("\n" + "="*60)
print("VISUALIZATION: PCA 2D Projection")
print("="*60)

# Project to 2D for visualization
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

# Create a simple text-based scatter plot (since we're in terminal)
print(f"\nFirst 5 patients' 2D coordinates:")
viz_df = pd.DataFrame({
    'patient_id': patient_img_df['patient_id'].values,
    'PC1': X_2d[:, 0],
    'PC2': X_2d[:, 1]
})
print(viz_df.head())

# Save for later plotting
viz_df.to_csv("../output_features/pca_2d_projection.csv", index=False)
print(f"\n✓ 2D projection saved for visualization")
print(f"  PC1 range: [{X_2d[:, 0].min():.1f}, {X_2d[:, 0].max():.1f}]")
print(f"  PC2 range: [{X_2d[:, 1].min():.1f}, {X_2d[:, 1].max():.1f}]")


VISUALIZATION: PCA 2D Projection

First 5 patients' 2D coordinates:
     patient_id        PC1        PC2
0  TCGA-BA-4074 -15.910409 -32.195136
1  TCGA-BA-4077 -43.795905  -7.769348
2  TCGA-BA-5153 -35.475531  -9.930598
3  TCGA-BA-5556 -21.686819 -34.807791
4  TCGA-BA-5557  -3.708504 -46.980573

✓ 2D projection saved for visualization
  PC1 range: [-68.0, 89.7]
  PC2 range: [-61.2, 80.9]


## MULTI-MODAL INTEGRATION

In [18]:
print("\n" + "="*60)
print("MULTI-MODAL INTEGRATION")
print("="*60)

# Load common samples
with open(COMMON_SAMPLES_FILE, 'r') as f:
    common_samples = set(line.strip() for line in f if line.strip())
print(f"\nCommon samples (target): {len(common_samples)}")

# ---- 4a: Load Omics Data ----
print("\n--- Loading Omics Data ---")

# Epigenetic data
print(f"\nLoading epigenetic data: {EPIGENETIC_FILE}")
epi_df = pd.read_csv(EPIGENETIC_FILE, index_col=0)
print(f"  Shape: {epi_df.shape}")

# Transpose to patients × genes
epi_patients = epi_df.T
epi_patients.index.name = 'patient_id'
epi_patients = epi_patients.reset_index()

# Clean patient IDs (remove -01 suffix, keep -11 for normals)
def clean_sample_id(sid):
    """Extract patient ID from sample ID"""
    if sid.endswith('-11'):
        return None  # Skip normal samples
    return sid[:12]



MULTI-MODAL INTEGRATION

Common samples (target): 82

--- Loading Omics Data ---

Loading epigenetic data: ../combined_epigenetic_score_matrix.csv
  Shape: (19199, 88)


In [19]:
epi_patients['patient_id'] = epi_patients['patient_id'].apply(clean_sample_id)
epi_patients = epi_patients[epi_patients['patient_id'].notna()]
epi_patients = epi_patients.groupby('patient_id').mean().reset_index()

# Add prefix to gene columns
gene_cols = [c for c in epi_patients.columns if c != 'patient_id']
epi_patients.columns = ['patient_id'] + [f'epi_{c}' for c in gene_cols]

print(f"  After cleaning: {epi_patients.shape}")
print(f"  Unique patients: {epi_patients['patient_id'].nunique()}")

# Mutation-Epigenetic Interaction data
print(f"\nLoading mutation interaction data: {MUTATION_INTERACTION_FILE}")
mut_df = pd.read_csv(MUTATION_INTERACTION_FILE, index_col=0)
print(f"  Shape: {mut_df.shape}")

# Transpose
mut_patients = mut_df.T
mut_patients.index.name = 'patient_id'
mut_patients = mut_patients.reset_index()

# Clean patient IDs
mut_patients['patient_id'] = mut_patients['patient_id'].apply(clean_sample_id)
mut_patients = mut_patients[mut_patients['patient_id'].notna()]
mut_patients = mut_patients.groupby('patient_id').mean().reset_index()

# Add prefix
mut_cols = [c for c in mut_patients.columns if c != 'patient_id']
mut_patients.columns = ['patient_id'] + [f'mut_{c}' for c in mut_cols]

print(f"  After cleaning: {mut_patients.shape}")
print(f"  Unique patients: {mut_patients['patient_id'].nunique()}")

  After cleaning: (82, 19200)
  Unique patients: 82

Loading mutation interaction data: ../mutation_epigenetic_interaction_matrix.csv
  Shape: (5594, 82)
  After cleaning: (82, 5595)
  Unique patients: 82


In [20]:
print(f"\n--- Loading Clinical Data ---")
print(f"File: {CLINICAL_FILE}")
clinical_df = pd.read_csv(CLINICAL_FILE, low_memory=False)
print(f"  Shape: {clinical_df.shape}")

# Use cases.submitter_id as patient_id
if 'cases.submitter_id' in clinical_df.columns:
    clinical_df['patient_id'] = clinical_df['cases.submitter_id'].str[:12]
    print(f"  Using cases.submitter_id for patient mapping")
elif 'Case_ID_12' in clinical_df.columns:
    clinical_df['patient_id'] = clinical_df['Case_ID_12']
    print(f"  Using Case_ID_12 for patient mapping")

# Aggregate clinical per patient (take first row since most info is repeated)
clinical_patients = clinical_df.groupby('patient_id').first().reset_index()
print(f"  After aggregation: {clinical_patients.shape}")
print(f"  Unique patients: {clinical_patients['patient_id'].nunique()}")


--- Loading Clinical Data ---
File: ../csv/Pathology_Cohorts_v7.csv
  Shape: (3651, 63)
  Using cases.submitter_id for patient mapping
  After aggregation: (82, 64)
  Unique patients: 82


### MERGING ALL MODALITIES

In [21]:
print("\n--- Merging All Modalities ---")
print(f"\nImage PCA features: {pca_df.shape}")
print(f"Epigenetic features: {epi_patients.shape}")
print(f"Mutation features: {mut_patients.shape}")
print(f"Clinical features: {clinical_patients.shape}")

# Find intersection of patients
image_patients = set(pca_df['patient_id'])
epi_patients_set = set(epi_patients['patient_id'])
mut_patients_set = set(mut_patients['patient_id'])
clinical_patients_set = set(clinical_patients['patient_id'])
common_patients_set = set(common_samples)

print(f"\nPatient overlap:")
print(f"  Image patients: {len(image_patients)}")
print(f"  Epigenetic patients: {len(epi_patients_set)}")
print(f"  Mutation patients: {len(mut_patients_set)}")
print(f"  Clinical patients: {len(clinical_patients_set)}")
print(f"  Common samples list: {len(common_patients_set)}")

# Find patients in all sets
all_patients = image_patients & epi_patients_set & mut_patients_set & clinical_patients_set & common_patients_set
all_patients = sorted(all_patients)

print(f"\n  ✓ Patients with ALL data types: {len(all_patients)}")

# Merge all modalities
integrated_df = pca_df[pca_df['patient_id'].isin(all_patients)].copy()

# Merge epigenetic
integrated_df = integrated_df.merge(
    epi_patients[epi_patients['patient_id'].isin(all_patients)],
    on='patient_id', how='inner'
)

# Merge mutation
integrated_df = integrated_df.merge(
    mut_patients[mut_patients['patient_id'].isin(all_patients)],
    on='patient_id', how='inner'
)

# Merge clinical
integrated_df = integrated_df.merge(
    clinical_patients[clinical_patients['patient_id'].isin(all_patients)],
    on='patient_id', how='inner'
)

print(f"\nFinal integrated shape: {integrated_df.shape}")
print(f"  Rows (patients): {len(integrated_df)}")
print(f"  Columns (features): {len(integrated_df.columns)}")

# Count features per modality
img_cols = [c for c in integrated_df.columns if c.startswith('img_pca_')]
epi_cols_final = [c for c in integrated_df.columns if c.startswith('epi_')]
mut_cols_final = [c for c in integrated_df.columns if c.startswith('mut_')]
clinical_cols_final = [c for c in integrated_df.columns if not c.startswith(('img_', 'epi_', 'mut_')) and c != 'patient_id']

print(f"\nFeature breakdown:")
print(f"  Image PCA features: {len(img_cols)}")
print(f"  Epigenetic features: {len(epi_cols_final)}")
print(f"  Mutation features: {len(mut_cols_final)}")
print(f"  Clinical features: {len(clinical_cols_final)}")
print(f"  TOTAL: {len(img_cols) + len(epi_cols_final) + len(mut_cols_final) + len(clinical_cols_final)}")

# Save
integrated_df.to_csv(OUTPUT_INTEGRATED, index=False)
print(f"\n✓ Integrated features saved to: {OUTPUT_INTEGRATED}")



--- Merging All Modalities ---

Image PCA features: (82, 83)
Epigenetic features: (82, 19200)
Mutation features: (82, 5595)
Clinical features: (82, 64)

Patient overlap:
  Image patients: 82
  Epigenetic patients: 82
  Mutation patients: 82
  Clinical patients: 82
  Common samples list: 82

  ✓ Patients with ALL data types: 82

Final integrated shape: (82, 24939)
  Rows (patients): 82
  Columns (features): 24939

Feature breakdown:
  Image PCA features: 82
  Epigenetic features: 19199
  Mutation features: 5594
  Clinical features: 63
  TOTAL: 24938

✓ Integrated features saved to: ../output_features/integrated_multimodal_features.csv


In [22]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"""
Pipeline Complete! Here's what we created:

1. Patient-level Image Features (PCA):
   - File: {OUTPUT_PATCH_FEATURES}
   - Shape: {pca_df.shape}
   - Features: {N_PCA_COMPONENTS} PCA components

2. Integrated Multi-Modal Features:
   - File: {OUTPUT_INTEGRATED}
   - Shape: {integrated_df.shape}
   - Patients: {len(all_patients)}
   - Modalities:
     • Image: {len(img_cols)} PCA features
     • Epigenetic: {len(epi_cols_final)} genes
     • Mutation: {len(mut_cols_final)} genes
     • Clinical: {len(clinical_cols_final)} features

Ready for ML modeling! 🚀
""")


SUMMARY

Pipeline Complete! Here's what we created:

1. Patient-level Image Features (PCA):
   - File: ../output_features/patient_image_features_pca.csv
   - Shape: (82, 83)
   - Features: 82 PCA components

2. Integrated Multi-Modal Features:
   - File: ../output_features/integrated_multimodal_features.csv
   - Shape: (82, 24939)
   - Patients: 82
   - Modalities:
     • Image: 82 PCA features
     • Epigenetic: 19199 genes
     • Mutation: 5594 genes
     • Clinical: 63 features

Ready for ML modeling! 🚀

